In [1]:
print("test")
from platform import python_version

print(python_version())

test
3.11.11


In [2]:
import sys

# Confirm that we're using Python 3
assert sys.version_info.major == 3, 'Oops, not running Python 3. Use Runtime > Change runtime type'

# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt
import os
import subprocess

print('TensorFlow version: {}'.format(tf.__version__))

2025-06-09 12:49:56.546318: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-09 12:49:56.546358: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-09 12:49:56.547508: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-09 12:49:56.554832: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-09 12:49:57.157266: W tensorflow/comp

TensorFlow version: 2.15.0


In [3]:
# TensorFlow and tf.keras
import tensorflow as tf
from tensorflow import keras


fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# scale the values to 0.0 to 1.0
train_images = train_images / 255.0
test_images = test_images / 255.0

# reshape for feeding into the model
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1)
test_images = test_images.reshape(test_images.shape[0], 28, 28, 1)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print('\ntrain_images.shape: {}, of {}'.format(train_images.shape, train_images.dtype))
print('test_images.shape: {}, of {}'.format(test_images.shape, test_images.dtype))


train_images.shape: (60000, 28, 28, 1), of float64
test_images.shape: (10000, 28, 28, 1), of float64


In [4]:
import numpy as np

# Numero totale di classi (Fashion MNIST ha 10 classi: 0–9)
NUM_CLASSES = 10
HALF = NUM_CLASSES // 2   # =5

# Primo sottoinsieme di classi: {0,1,2,3,4}
classes_1 = list(range(0, HALF))
# Secondo sottoinsieme di classi: {5,6,7,8,9}
classes_2 = list(range(HALF, NUM_CLASSES))

# --- Funzione per estrarre esempi la cui label è in `class_list`, rimappando le etichette su 0..(len(class_list)-1) ---
def extract_subset(images, labels, class_list):
    """
    Filtra images, labels tenendo solo le etichette presenti in class_list;
    rimappa le etichette in un range 0..(len(class_list)-1).
    """
    mask = np.isin(labels, class_list)
    imgs_sub = images[mask]
    lbls_sub = labels[mask]
    # Rimappo ogni label x in label_index[x] dove label_index mappa class_list su 0..len-1
    label_index = {c:i for i,c in enumerate(class_list)}
    lbls_sub_mapped = np.vectorize(lambda x: label_index[x])(lbls_sub)
    return imgs_sub, lbls_sub_mapped

# Estrazione per il “client 1” (classi 0–4)
train_images_1, train_labels_1 = extract_subset(train_images, train_labels, classes_1)
test_images_1,  test_labels_1  = extract_subset(test_images,  test_labels,  classes_1)

# Estrazione per il “client 2” (classi 5–9)
train_images_2, train_labels_2 = extract_subset(train_images, train_labels, classes_2)
test_images_2,  test_labels_2  = extract_subset(test_images,  test_labels,  classes_2)

print("Client 1 – esempi di train:", train_images_1.shape, train_labels_1.shape)
print("Client 2 – esempi di train:", train_images_2.shape, train_labels_2.shape)


Client 1 – esempi di train: (30000, 28, 28, 1) (30000,)
Client 2 – esempi di train: (30000, 28, 28, 1) (30000,)


In [5]:
import tensorflow as tf
from tensorflow import keras

def create_local_model(num_local_classes=5):
    """
    Restituisce un modello convolutional semplice con output = num_local_classes.
    """
    m = keras.Sequential([
        keras.layers.Conv2D(input_shape=(28,28,1), filters=8, kernel_size=3, 
                            strides=2, activation='relu', name='Conv1'),
        keras.layers.Flatten(),
        keras.layers.Dense(num_local_classes, activation='softmax', name='Softmax')
    ])
    return m

# Creo i due modelli
model_1 = create_local_model(num_local_classes=HALF)   # output=5, per classi [0-4]
model_2 = create_local_model(num_local_classes=HALF)   # output=5, per classi [5-9]

print("=== Model 1 (classi 0..4) ===")
model_1.summary()
print("\n=== Model 2 (classi 5..9) ===")
model_2.summary()


=== Model 1 (classi 0..4) ===
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Conv1 (Conv2D)              (None, 13, 13, 8)         80        
                                                                 
 flatten (Flatten)           (None, 1352)              0         
                                                                 
 Softmax (Dense)             (None, 5)                 6765      
                                                                 
Total params: 6845 (26.74 KB)
Trainable params: 6845 (26.74 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________

=== Model 2 (classi 5..9) ===
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Conv1 (Conv2D)              (None, 13, 13, 8)         80        
     

2025-06-09 12:49:58.831054: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-09 12:49:58.871836: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-09 12:49:58.874107: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [6]:
# Numero di epoche
EPOCHS_LOCAL = 10   # puoi aumentare se vuoi

# ---> Model 1 su train_images_1 / train_labels_1 <---
model_1.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
model_1.fit(train_images_1, train_labels_1, epochs=EPOCHS_LOCAL, validation_split=0.1)
loss1, acc1 = model_1.evaluate(test_images_1, test_labels_1)
print(f"\nModel1 (classi 0–4) – Test accuracy: {acc1:.4f}")

# ---> Model 2 su train_images_2 / train_labels_2 <---
model_2.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
model_2.fit(train_images_2, train_labels_2, epochs=EPOCHS_LOCAL, validation_split=0.1)
loss2, acc2 = model_2.evaluate(test_images_2, test_labels_2)
print(f"\nModel2 (classi 5–9) – Test accuracy: {acc2:.4f}")


Epoch 1/10


2025-06-09 12:49:59.708511: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:447] Loaded runtime CuDNN library: 8.6.0 but source was compiled with: 8.9.4.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building from sources, make sure the library loaded at runtime is compatible with the version specified during compile configuration.
2025-06-09 12:49:59.709202: W tensorflow/core/framework/op_kernel.cc:1839] OP_REQUIRES failed at conv_ops_fused_impl.h:625 : UNIMPLEMENTED: DNN library is not found.


UnimplementedError: Graph execution error:

Detected at node sequential/Conv1/Relu defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/zerod/lab/jupyterlab/jupyter-extension-project/venv3.11/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/zerod/lab/jupyterlab/jupyter-extension-project/venv3.11/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/zerod/lab/jupyterlab/jupyter-extension-project/venv3.11/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/zerod/lab/jupyterlab/jupyter-extension-project/venv3.11/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/zerod/.local/share/uv/python/cpython-3.11.11-linux-x86_64-gnu/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/home/zerod/.local/sha

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_federated as tff
from tensorflow import keras
from tensorflow_federated.learning.models import from_keras_model
from tensorflow_federated.learning.algorithms import build_weighted_fed_avg
from tensorflow_federated.learning.optimizers import build_adam, build_sgdm

# 1) Definisci model_fn usando from_keras_model (non più tff.learning.from_keras_model)
def model_fn():
    keras_model = create_local_model(num_local_classes=NUM_CLASSES)
    return from_keras_model(
        keras_model=keras_model,
        input_spec=input_spec,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
    )  # :contentReference[oaicite:0]{index=0}

# 2) Costruisci il processo di FedAvg con la nuova API
iterative_process = build_weighted_fed_avg(
    model_fn=model_fn,
    client_optimizer_fn=lambda: build_adam(learning_rate=0.01),
    server_optimizer_fn=lambda: build_sgdm(learning_rate=1.0),
)  # :contentReference[oaicite:1]{index=1}

# 3) Inizializza e fai girare i round
state = iterative_process.initialize()
NUM_ROUNDS = 20

for rnd in range(1, NUM_ROUNDS + 1):
    state, metrics = iterative_process.next(state, [train_dataset_1, train_dataset_2])
    print(f"Round {rnd:02d} — loss={metrics.loss:.4f}, acc={metrics.sparse_categorical_accuracy:.4f}")

# 4) Estrai i pesi e assegnali al tuo Keras model
global_model = create_local_model(num_local_classes=NUM_CLASSES)
model_weights = iterative_process.get_model_weights(state)
model_weights.assign_weights_to(global_model)  # :contentReference[oaicite:2]{index=2}

# Ora `global_model` è il modello federato addestrato su tutte e 10 le classi.
